# 10장. 분류 분석으로 주문 취소 여부 예측하기

이 노트북은 완료 주문과 취소 주문만 사용해 `is_cancelled`를 예측하는 이진 분류 실습입니다.

## 학습 목표

- `completed=0`, `cancelled=1`로 타깃 범위를 명확하게 정의합니다.
- `refunded`와 기타 상태를 이진 분류 대상에서 제외합니다.
- 병합의 키 관계, 행 수, 미매칭을 검증합니다.
- train, validation, test를 분리합니다.
- Dummy 기준 모델, Logistic Regression, Random Forest를 비교합니다.
- validation에서 모델과 임계값을 선택하고 test는 최종 평가에 한 번만 사용합니다.
- 예측 결과에서 고객명 등 불필요한 개인정보를 제외합니다.


## 1. 프로젝트 루트와 실행 환경 설정

전처리 파일이 없다면 프로젝트 루트에서 먼저 다음 명령을 실행하세요.

```bash
python scripts/preprocess_data.py
```


In [1]:
from pathlib import Path
import sys

import pandas as pd


def find_project_root(start_path):
    start_path = Path(start_path).resolve()
    for candidate in [start_path, *start_path.parents]:
        if (candidate / 'requirements.txt').exists() and (candidate / 'scripts').exists():
            return candidate
    raise FileNotFoundError('프로젝트 루트 폴더를 찾을 수 없습니다.')


PROJECT_ROOT = find_project_root(Path(r"C:\dev\llm-data-analysis-course-ch10"))
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
REPORT_DIR = PROJECT_ROOT / 'reports'
REPORT_DIR.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('Python 실행 파일:', sys.executable)
print('프로젝트 루트:', PROJECT_ROOT)
print('전처리 데이터 폴더:', PROCESSED_DIR)
print('보고서 폴더:', REPORT_DIR)


Python 실행 파일: c:\dev\llm-data-analysis-course\.venv\Scripts\python.exe
프로젝트 루트: C:\dev\llm-data-analysis-course-ch10
전처리 데이터 폴더: C:\dev\llm-data-analysis-course-ch10\data\processed
보고서 폴더: C:\dev\llm-data-analysis-course-ch10\reports


## 2. 전처리 데이터 확인과 불러오기

이 장에서는 `customers_clean.csv`, `orders_clean.csv`, `order_items_clean.csv`를 사용합니다.


In [2]:
required_files = [
    PROCESSED_DIR / 'customers_clean.csv',
    PROCESSED_DIR / 'orders_clean.csv',
    PROCESSED_DIR / 'order_items_clean.csv',
]

missing_files = [path for path in required_files if not path.exists()]
if missing_files:
    raise FileNotFoundError(
        '전처리 파일이 없습니다: '
        + ', '.join(str(path) for path in missing_files)
        + '. 먼저 python scripts/preprocess_data.py를 실행하세요.'
    )

customers = pd.read_csv(PROCESSED_DIR / 'customers_clean.csv')
orders = pd.read_csv(PROCESSED_DIR / 'orders_clean.csv')
order_items = pd.read_csv(PROCESSED_DIR / 'order_items_clean.csv')

print('customers:', customers.shape, customers.columns.tolist())
print('orders:', orders.shape, orders.columns.tolist())
print('order_items:', order_items.shape, order_items.columns.tolist())


customers: (150, 6) ['customer_id', 'name', 'gender', 'age', 'city', 'signup_date']
orders: (300, 7) ['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status', 'order_month', 'order_dayofweek']
order_items: (764, 6) ['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price', 'line_total']


## 3. 주문 상태와 타깃 범위 확인

이번 실습은 완료 주문과 취소 주문만 비교합니다.

- `completed` → 0
- `cancelled` → 1
- `refunded`와 기타 상태 → 학습 대상에서 제외


In [3]:
status_summary = (
    orders['order_status']
    .value_counts(dropna=False)
    .rename_axis('order_status')
    .reset_index(name='order_count')
)
status_summary


,order_status,order_count
0,completed,184
1,cancelled,64
2,refunded,52


## 4. 분류 데이터와 병합 검증표 생성

공통 함수는 주문 특징 집계, 안전한 병합, 시간 순서 점검, 타깃 생성까지 수행합니다.


In [4]:
from src.classification import (
    build_classification_dataset,
    target_distribution,
)

(
    model_data,
    numeric_features,
    categorical_features,
    merge_checks,
    data_quality_checks,
) = build_classification_dataset(
    customers=customers,
    orders=orders,
    order_items=order_items,
)

target_dist = target_distribution(model_data)

print('모델링 데이터:', model_data.shape)
print('숫자형 feature:', numeric_features)
print('범주형 feature:', categorical_features)
display(target_dist)
display(merge_checks)
display(data_quality_checks)


모델링 데이터: (248, 16)
숫자형 feature: ['age', 'item_count', 'total_quantity', 'order_amount', 'order_month', 'order_dayofweek']
범주형 feature: ['gender', 'city', 'payment_method']


,is_cancelled,class_label,count,ratio
0,0,completed,184,0.7419
1,1,cancelled,64,0.2581


,merge,before_rows,after_rows,row_count_preserved,unmatched_count,status
0,order_id → order_item_features,248,248,True,0,PASS
1,customer_id → customers,248,248,True,0,PASS


,check_item,count
0,binary_target_rows,248
1,completed_rows,184
2,cancelled_rows,64
3,excluded_status_rows,52
4,line_total_mismatch,0
5,negative_days_since_signup,45
6,missing_order_item_features,0
7,missing_customer_match,0
8,status_scope:completed,184
9,status_scope:cancelled,64


### 데이터 품질 오류 및 코드 수정 기록

**1. 발견한 문제**

고객 가입일이 주문일보다 늦은 주문 45건(완료 35건, 취소 10건)이 확인되어 공식 Notebook 실행이 중단되었습니다.

**2. 나의 판단**

이번 분석의 목적은 주문 취소 여부를 예측하는 것입니다. 고객 가입 후 경과 기간보다는 주문 금액, 상품 수량, 결제 방법 등이 취소 여부와 더 관련이 있을 것으로 판단했습니다.

가입 경과일의 예측 영향이 크지 않을 것으로 예상했고, 날짜가 불일치하는 데이터도 다수 확인되어 해당 Feature를 제외하기로 했습니다.

**3. 수정 내용**

원본 데이터는 변경하지 않고, 작업용 `classification.py`에서 `days_since_signup`을 모델 입력 Feature에서 제외했습니다. 날짜 역전 45건은 데이터 품질 검증 결과에 그대로 기록했습니다.

**4. 한계**

가입 경과일의 실제 예측 영향은 검증하지 않았습니다. 향후 해당 변수를 포함한 모델과 비교하면 제외 판단의 적절성을 확인할 수 있습니다. 또한 날짜 불일치의 원인은 추가 확인이 필요합니다.

## 5. 데이터 누수 확인

`order_status`와 `is_cancelled`는 입력 feature에 포함되면 안 됩니다.


In [5]:
features = numeric_features + categorical_features
leakage_columns = {'order_status', 'is_cancelled'}
leakage_found = sorted(leakage_columns.intersection(features))

if leakage_found:
    raise ValueError(f'데이터 누수 위험 컬럼이 포함되었습니다: {leakage_found}')

print('사용 feature:', features)
print('데이터 누수 컬럼 없음')


사용 feature: ['age', 'item_count', 'total_quantity', 'order_amount', 'order_month', 'order_dayofweek', 'gender', 'city', 'payment_method']
데이터 누수 컬럼 없음


## 6. train, validation, test 분리

모델과 임계값은 validation에서 선택하고, test는 최종 평가에 한 번만 사용합니다.


In [6]:
from src.classification import (
    split_train_validation_test,
    build_split_summary,
)

(
    X_train,
    X_valid,
    X_test,
    y_train,
    y_valid,
    y_test,
    features,
) = split_train_validation_test(
    model_data=model_data,
    numeric_features=numeric_features,
    categorical_features=categorical_features,
    random_state=42,
)

split_summary = build_split_summary(y_train, y_valid, y_test)

print('X_train:', X_train.shape)
print('X_valid:', X_valid.shape)
print('X_test:', X_test.shape)
display(split_summary)


X_train: (148, 9)
X_valid: (50, 9)
X_test: (50, 9)


,split,is_cancelled,class_label,count,ratio
0,train,0,completed,110,0.7432
1,train,1,cancelled,38,0.2568
2,validation,0,completed,37,0.7400
3,validation,1,cancelled,13,0.2600
4,test,0,completed,37,0.7400
5,test,1,cancelled,13,0.2600


## 7. 기준 모델과 학습 모델 비교

가장 많은 클래스만 예측하는 Dummy 모델을 기준으로 Logistic Regression과 Random Forest의 validation 성능을 비교합니다.


In [7]:
from src.classification import train_and_compare_on_validation

(
    models,
    validation_comparison,
    validation_predictions,
    validation_probabilities,
) = train_and_compare_on_validation(
    X_train=X_train,
    X_valid=X_valid,
    y_train=y_train,
    y_valid=y_valid,
    numeric_features=numeric_features,
    categorical_features=categorical_features,
    random_state=42,
)

validation_comparison


,model,evaluation_split,accuracy,precision,recall,f1
0,Logistic Regression,validation,0.72,0.466667,0.538462,0.500000
1,Random Forest,validation,0.68,0.363636,0.307692,0.333333
2,Dummy Most Frequent,validation,0.74,0.000000,0.000000,0.000000


## 8. validation에서 모델과 임계값 선택

Dummy 모델은 기준선이므로 실제 선택 후보에서는 제외합니다. validation의 f1-score를 우선하고 recall과 precision을 함께 확인합니다.


In [8]:
from src.classification import threshold_metrics, choose_threshold

non_dummy_comparison = validation_comparison[
    validation_comparison['model'] != 'Dummy Most Frequent'
]

selected_model_name = str(non_dummy_comparison.iloc[0]['model'])
selected_model = models[selected_model_name]
validation_proba = validation_probabilities[selected_model_name]

threshold_df = threshold_metrics(y_valid, validation_proba)
selected_threshold = choose_threshold(threshold_df)

print('선택 모델:', selected_model_name)
print('선택 임계값:', selected_threshold)
display(threshold_df.sort_values(['f1', 'recall'], ascending=False).head(10))


선택 모델: Logistic Regression
선택 임계값: 0.45


,threshold,accuracy,precision,recall,f1
5,0.45,0.70,0.450000,0.692308,0.545455
8,0.60,0.78,0.600000,0.461538,0.521739
4,0.40,0.62,0.384615,0.769231,0.512821
6,0.50,0.72,0.466667,0.538462,0.500000
7,0.55,0.76,0.545455,0.461538,0.500000
3,0.35,0.52,0.333333,0.846154,0.478261
2,0.30,0.44,0.307692,0.923077,0.461538
1,0.25,0.34,0.282609,1.000000,0.440678
0,0.20,0.30,0.270833,1.000000,0.426230
9,0.65,0.74,0.500000,0.307692,0.380952


## 9. test 데이터 최종 평가

이제 선택이 끝났으므로 test 데이터에서 최종 성능을 한 번 확인합니다.


In [9]:
from src.classification import (
    final_test_evaluation,
    confusion_matrix_dataframe,
    classification_report_dataframe,
)

y_pred_test, y_proba_test, test_metrics = final_test_evaluation(
    selected_model,
    X_test,
    y_test,
    threshold=selected_threshold,
)

confusion_df = confusion_matrix_dataframe(y_test, y_pred_test)
classification_report_df = classification_report_dataframe(y_test, y_pred_test)

display(test_metrics)
display(confusion_df)
display(classification_report_df)


,evaluation_split,threshold,selection_status,accuracy,precision,recall,f1
0,test,0.45,frozen_before_test,0.6,0.315789,0.461538,0.375


,pred_completed,pred_cancelled
actual_completed,24,13
actual_cancelled,7,6


,label,precision,recall,f1-score,support
0,completed,0.774194,0.648649,0.705882,37.0
1,cancelled,0.315789,0.461538,0.375000,13.0
2,accuracy,0.600000,0.600000,0.600000,0.6
3,macro avg,0.544992,0.555094,0.540441,50.0
4,weighted avg,0.655008,0.600000,0.619853,50.0


혼동행렬은 다음처럼 읽습니다.

| 구분 | 의미 |
|---|---|
| True Negative | 실제 완료 주문을 완료로 예측 |
| False Positive | 실제 완료 주문을 취소로 잘못 예측 |
| False Negative | 실제 취소 주문을 완료로 잘못 예측 |
| True Positive | 실제 취소 주문을 취소로 예측 |


## 10. 개인정보를 제외한 예측 결과 생성

고객명, 이메일, 주소 같은 불필요한 식별정보는 결과 파일에 저장하지 않습니다.


In [10]:
from src.classification import create_prediction_result

prediction_result = create_prediction_result(
    source_index=X_test.index,
    y_test=y_test,
    y_pred=y_pred_test,
    y_proba=y_proba_test,
    model_name=selected_model_name,
    threshold=selected_threshold,
)

prediction_result.head()


,record_id,source_index,actual_is_cancelled,predicted_is_cancelled,cancel_probability,model,threshold
0,test_0001,28,0,1,0.683535,Logistic Regression,0.45
1,test_0002,146,0,0,0.409431,Logistic Regression,0.45
2,test_0003,211,0,1,0.512531,Logistic Regression,0.45
3,test_0004,125,0,1,0.602233,Logistic Regression,0.45
4,test_0005,77,0,1,0.746377,Logistic Regression,0.45


## 11. LLM 코드 검토 체크리스트


In [11]:
from src.classification import build_classification_checklist

classification_checklist = build_classification_checklist()
classification_checklist


,check_item,status
0,completed와 cancelled만 사용해 이진 타깃을 만들었는가?,□
1,refunded 등 다른 상태를 0 클래스에 섞지 않았는가?,□
2,예측 시점과 feature 가용성 가정을 설명할 수 있는가?,□
3,line_total = quantity × unit_price를 검증했는가?,□
4,병합에 validate를 사용하고 미매칭 0건을 확인했는가?,□
5,"order_status, target, ID, 사후 정보를 feature에서 제외했는가?",□
6,"train, validation, test를 분리했는가?",□
7,모델과 threshold는 validation에서 선택했는가?,□
8,모델과 threshold를 고정한 뒤 test를 한 번만 사용했는가?,□
9,Dummy 기준 모델과 비교했는가?,□


## 12. 전체 파이프라인 실행과 산출물 저장

위 과정을 한 번에 다시 실행해 CSV와 Markdown 보고서를 생성합니다.


In [12]:
from src.classification import run_classification_analysis

classification_result = run_classification_analysis(
    processed_dir=PROCESSED_DIR,
    report_dir=REPORT_DIR,
    random_state=42,
)

print('선택 모델:', classification_result['selected_model_name'])
print('선택 임계값:', classification_result['selected_threshold'])
display(classification_result['test_metrics'])

for name, path in classification_result['output_paths'].items():
    print(name, path, 'OK' if path.exists() else 'MISSING')


선택 모델: Logistic Regression
선택 임계값: 0.45


,evaluation_split,threshold,selection_status,accuracy,precision,recall,f1
0,test,0.45,frozen_before_test,0.6,0.315789,0.461538,0.375


model_data_internal C:\dev\llm-data-analysis-course-ch10\reports\ch10_classification_model_data_internal.csv OK
target_distribution C:\dev\llm-data-analysis-course-ch10\reports\ch10_target_distribution.csv OK
feature_audit C:\dev\llm-data-analysis-course-ch10\reports\ch10_feature_audit.csv OK
merge_checks C:\dev\llm-data-analysis-course-ch10\reports\ch10_merge_checks.csv OK
data_quality_checks C:\dev\llm-data-analysis-course-ch10\reports\ch10_data_quality_checks.csv OK
split_summary C:\dev\llm-data-analysis-course-ch10\reports\ch10_split_summary.csv OK
validation_comparison C:\dev\llm-data-analysis-course-ch10\reports\ch10_validation_model_comparison.csv OK
threshold_metrics C:\dev\llm-data-analysis-course-ch10\reports\ch10_validation_threshold_metrics.csv OK
test_metrics C:\dev\llm-data-analysis-course-ch10\reports\ch10_test_metrics.csv OK
predictions_internal C:\dev\llm-data-analysis-course-ch10\reports\ch10_classification_predictions_internal.csv OK
predictions C:\dev\llm-data-analy

## 13. LLM 검토 프롬프트 예시

```text
온라인 쇼핑몰 주문 취소 분류 코드를 검토해 주세요.

타깃 기준:
- completed=0
- cancelled=1
- refunded와 기타 상태는 제외

검토 항목:
1. order_status와 is_cancelled가 feature에서 제외되었는가?
2. 병합에 validate와 indicator가 사용되었는가?
3. train, validation, test가 분리되었는가?
4. 모델과 임계값은 validation에서 선택했는가?
5. test는 최종 평가에 한 번만 사용했는가?
6. Dummy 기준 모델보다 나은가?
7. accuracy, precision, recall, f1을 함께 확인했는가?
8. 예측 결과에 개인정보가 포함되지 않았는가?
9. 모델 결과를 취소 원인으로 단정하지 않았는가?

필수 수정과 권장 개선을 구분해 설명해 주세요.
```


## 14. 정리

이번 장에서는 완료 주문과 취소 주문을 명확하게 구분하고, 병합 검증, 데이터 누수 방지, train/validation/test 분리, 기준 모델 비교, validation 기반 모델·임계값 선택, 최종 test 평가를 수행했습니다.

다음 장에서는 LLM을 활용해 분석 질문과 프롬프트를 더 구체적으로 만드는 방법으로 이어집니다.


# Chapter 10 답안 양식. 분류 분석으로 주문 취소 여부 예측하기

## 제출 정보
- 이름: 오현우
- GitHub ID: dhgusdn0312
- 작성일: 2026.11.18
- 최종 Notebook URL: https://github.com/dhgusdn0312/llm-data-analysis-study/blob/main/chapter10/chapter10.ipynb

## 1. 분류 문제 정의

- 타깃: `is_cancelled`
- 0의 의미: 완료 주문 (`completed`)
- 1의 의미: 취소 주문 (`cancelled`)
- 제외한 상태: `refunded` 및 기타 상태
- 예측 단위: 주문 1건
- 예측 시점: 주문 접수 직후, 취소 여부가 확정되기 전
- 예측 시점에 사용할 수 있는 정보: 주문 금액, 상품 수량, 결제 방법, 주문일, 고객 기본 정보
- 예측 이후에만 알 수 있어 제외한 정보: 주문 상태, 취소 사유, 취소 일시

### 나의 해석과 판단
주문 취소 여부를 미리 예측하는 것이 목적이므로 완료와 취소 주문만 구분했습니다. 취소 이후에 알 수 있는 정보는 예측에 사용할 수 없으므로 제외했습니다.

---

## 2. Feature Contract와 Leakage Audit

### 사용한 숫자형 Feature
`age`, `item_count`, `total_quantity`, `order_amount`, `order_month`, `order_dayofweek`

### 사용한 범주형 Feature
`gender`, `city`, `payment_method`

### 금지 Feature

```text
order_status
is_cancelled
order_id
customer_id
product_id
cancel_reason
cancelled_at
취소 이후 정보
```

### 나의 해석과 판단
데이터에 컬럼이 있어도 예측 시점에 알 수 없다면 사용하면 안 됩니다. 정답에 해당하는 정보가 포함되면 실제보다 좋은 결과가 나올 수 있기 때문입니다.

가입 경과일(`days_since_signup`)은 취소 여부에 큰 영향을 주지 않을 것으로 생각했고, 날짜 오류도 있어 Feature에서 제외했습니다. 실제 영향은 추가 확인이 필요합니다.

주문 금액과 수량은 주문 접수 시 확인할 수 있다고 가정했습니다.

---

## 3. 주문 단위 특징과 병합 검증

### 계산 관계 확인

```text
line_total = quantity × unit_price
```

- 계산 불일치 건수: 0건
- 주문 특징 `order_id` 중복 건수: 0건

### 병합 검증

| 병합 | 관계 | 병합 전 행 수 | 병합 후 행 수 | 미매칭 | 판정 |
| --- | --- | ---: | ---: | ---: | --- |
| 주문 ↔ 주문 특징 | one_to_one | 248 | 248 | 0 | PASS |
| 주문 ↔ 고객 | many_to_one | 248 | 248 | 0 | PASS |

### 결과 관찰
병합 전후 주문 수가 248건으로 동일했고, 미매칭과 계산 불일치는 없었습니다. 다만 가입일이 주문일보다 늦은 주문 45건이 확인됐습니다.

### 나의 해석과 판단
미매칭을 무조건 0으로 채우면 실제 값이 0인 것인지 데이터가 없는 것인지 알 수 없습니다. 따라서 원인을 먼저 확인해야 합니다.

날짜 오류는 원본을 수정하지 않고 기록했으며, 가입 경과일은 Feature에서 제외했습니다.

---

## 4. 클래스 비율

| 클래스 | 의미 | 개수 | 비율 |
| ---: | --- | ---: | ---: |
| 0 | completed | 184 | 74.19% |
| 1 | cancelled | 64 | 25.81% |

![클래스 비율](images/step02_class_ratio.png)

### 결과 관찰
완료 주문 184건, 취소 주문 64건으로 완료 주문이 약 3배 많았습니다.

### 나의 해석과 판단
모든 주문을 완료로 예측해도 Accuracy가 약 74% 나올 수 있습니다. 따라서 취소 주문을 얼마나 잘 찾아내는지도 함께 확인해야 합니다.

---

## 5. Dummy Baseline과 Validation 모델 비교

| 모델 | Accuracy | Precision | Recall | F1 |
| --- | ---: | ---: | ---: | ---: |
| Dummy Most Frequent | 0.740 | 0.000 | 0.000 | 0.000 |
| Logistic Regression | 0.720 | 0.467 | 0.538 | 0.500 |
| Random Forest | 0.680 | 0.364 | 0.308 | 0.333 |

![Baseline 비교](images/step03_baseline.png)

- Validation에서 선택한 모델: Logistic Regression

### 결과 관찰
Dummy는 Accuracy가 가장 높지만 취소 주문을 하나도 맞히지 못했습니다. Logistic Regression은 Recall과 F1이 가장 높았습니다.

### 나의 해석과 판단
취소 주문을 미리 찾아내는 것이 목적이므로 Logistic Regression을 선택했습니다. Dummy보다 실제 취소 주문을 찾아낸다는 점에서 나았습니다.

### 한계와 추가 확인 사항
Precision이 약 46.7%로 낮은 편입니다. 다른 데이터에서도 비슷한 결과가 나오는지 확인이 필요합니다.

---

## 6. Train / Validation / Test 역할

| Split | 역할 | 클래스 0 건수 | 클래스 1 건수 |
| --- | --- | ---: | ---: |
| Train | 모델 학습 | 110 | 38 |
| Validation | 모델·Threshold 선택 | 37 | 13 |
| Test | 선택 이후 최종 평가 | 37 | 13 |

### 나의 해석과 판단
Train으로 학습하고 Validation에서 모델과 Threshold를 정했습니다. Test는 최종 평가에만 사용했습니다.

Test 결과를 보고 기준을 바꾸면 공정한 평가가 어려워지기 때문입니다.

### 교육용 random split의 한계
실제 운영 전에는 과거 주문으로 학습하고 이후 주문에서도 잘 예측하는지 확인할 필요가 있습니다.

---

## 7. Validation 성능과 우선 지표

선택 모델 Logistic Regression, Threshold 0.45 기준

- Accuracy: 0.700
- Precision: 0.450
- Recall: 0.692
- F1: 0.545

### 내가 가장 중요하게 본 지표
Recall과 F1

### 그 이유
취소 주문을 놓치면 미리 대응할 기회를 잃을 수 있으므로 FN을 줄이는 것이 중요하다고 생각했습니다.

다만 FP가 많아질 수도 있어 Precision과 F1도 함께 확인했습니다. 실제 업무 비용은 확인하지 못했으므로 가정에 따른 판단입니다.

---

## 8. Threshold 비교

| Threshold | Precision | Recall | F1 | 예상 특징 |
| ---: | ---: | ---: | ---: | --- |
| 0.40 | 0.385 | 0.769 | 0.513 | 취소 발견 증가, FP 증가 가능 |
| 0.45 | 0.450 | 0.692 | 0.545 | F1이 가장 높음 |
| 0.50 | 0.467 | 0.538 | 0.500 | Recall 감소, FN 증가 가능 |

![Threshold 비교](images/step05_threshold.png)

### 내가 선택한 Threshold
0.45

### 선택 이유
Validation에서 F1이 가장 높았고, 기본값 0.50보다 취소 주문을 더 많이 찾아냈습니다.

### 나의 해석과 판단
Test 결과를 보고 Threshold를 변경하면 공정한 평가가 되지 않으므로 Validation에서 미리 정했습니다.

---

## 9. FP/FN 해석

- FP 의미: 완료 주문을 취소 위험으로 잘못 예측한 경우 (13건)
- FN 의미: 취소 주문을 완료로 잘못 예측한 경우 (7건)
- 현재 목적에서 더 부담스러울 수 있는 오류: FN
- 그 판단에 추가로 필요한 업무 정보: 실제 취소 손실, 사전 대응 비용, 불필요한 고객 안내에 따른 비용

![Confusion Matrix](images/step06_confusion_matrix.png)

### 업무·분석적 의미
FN은 취소 위험을 놓쳐 미리 대응하지 못하는 경우입니다. FP는 정상 주문에 불필요한 확인이나 안내를 하게 되는 경우입니다.

취소를 미리 발견하는 것이 목적이므로 FN이 더 중요하다고 생각했습니다.

### 한계와 추가 확인 사항
실제 비용은 확인하지 못했습니다. 어떤 업무 조치를 하는지에 따라 FP가 더 중요할 수도 있습니다.

---

## 10. Final Test

- 고정한 모델: Logistic Regression
- 고정한 Threshold: 0.45
- Accuracy: 0.600
- Precision: 0.316
- Recall: 0.462
- F1: 0.375
- Validation 대비 변화: Accuracy -0.100, Precision -0.134, Recall -0.231, F1 -0.170

### 결과 관찰
Validation보다 모든 지표가 낮아졌습니다. 취소 주문 13건 중 7건을 놓쳤고, 완료 주문 13건을 취소로 잘못 예측했습니다.

### 나의 해석과 판단
Test 결과가 예상보다 낮아 실제 업무에 사용하기에는 부족하다고 생각합니다.

학습 데이터가 148건으로 적고, 현재 Feature만으로 취소 여부를 구분하기 어려웠을 수도 있습니다. 정확한 원인은 추가 확인이 필요합니다.

### 선택 보호 확인
- [x] Test 결과를 보기 전에 모델을 고정했습니다.
- [x] Test 결과를 보기 전에 Threshold를 고정했습니다.
- [x] Test 결과를 본 뒤 같은 Test를 이용해 선택을 바꾸지 않았습니다.

---

## 11. Internal / Public 결과 구분

### 공개 Prediction 컬럼

```text
record_id
actual_is_cancelled
predicted_is_cancelled
cancel_probability
model
threshold
```

### 공개 결과에 없어야 할 컬럼

```text
source_index
order_id
customer_id
product_id
```

### 결과 관찰
내부 결과에는 `source_index`가 있었지만 공개 CSV에서는 제거됐습니다. 자동 Validation에서도 PASS를 확인했습니다.

### 나의 해석과 판단
내부 분석에는 주문을 추적할 정보가 필요할 수 있지만, 공개 결과에는 불필요한 식별 정보를 넣지 않는 것이 적절하다고 생각합니다.

---

## 12. Validation Evidence

- [x] `ch10_target_distribution.csv`
- [x] `ch10_feature_audit.csv`
- [x] `ch10_merge_checks.csv`
- [x] `ch10_data_quality_checks.csv`
- [x] `ch10_split_summary.csv`
- [x] `ch10_validation_model_comparison.csv`
- [x] `ch10_validation_threshold_metrics.csv`
- [x] `ch10_test_metrics.csv`
- [x] `ch10_confusion_matrix.csv`
- [x] `ch10_classification_validation.csv`

### 자동 Validation 결과
총 7개 항목이 모두 PASS였습니다. Notebook과 전체 스크립트의 모델, Threshold, Test 결과도 일치했습니다.

### 자동 PASS가 의미하지 않는 것
자동 검증을 통과했다고 실제 업무에서도 잘 예측한다는 뜻은 아닙니다.

날짜 오류 45건이 남아 있고, 실제 운영에 사용하려면 시간에 따른 성능 변화와 고객별 예측 차이 등도 확인해야 합니다.

---

## 13. 최종 사용 판단

- [ ] 참고용 사용 가능
- [ ] 추가 검증 후 사용 가능
- [x] 현재 사용 보류

### 판단 근거
1. Dummy보다 취소 주문을 찾아내는 성능은 좋아졌지만 Test F1은 0.375였습니다.
2. 실제 취소 주문 13건 중 7건을 놓쳤고, FP도 13건 발생했습니다.
3. 데이터가 적고 날짜 오류도 있어 추가 검증이 필요합니다.

### 현재 모델의 가장 큰 위험
취소 주문을 충분히 찾아내지 못하면서 정상 주문에도 불필요한 경고를 발생시키는 것입니다.

### 다음 개선 우선순위
1. 취소 여부와 관련된 Feature 추가 검토
2. 데이터 확대 및 날짜 오류 확인
3. 시간순 데이터로 예측 성능 확인

### 예측과 인과 구분
이번 분석에서는 주문 정보를 이용해 취소 여부를 예측했습니다. 하지만 어떤 정보가 실제 취소를 일으킨 원인인지는 알 수 없습니다.

---

## 최종 체크

- [x] `completed=0`, `cancelled=1` 타깃 범위를 정의했습니다.
- [x] `refunded`와 기타 상태를 제외했습니다.
- [x] 예측 시점과 Feature Contract를 정의했습니다.
- [x] target, ID, 사후 정보를 feature에서 제외했습니다.
- [x] `line_total = quantity × unit_price`를 검증했습니다.
- [x] 주문 단위 특징과 병합 관계를 검증했습니다.
- [x] 클래스 비율을 확인했습니다.
- [x] Dummy baseline과 비교했습니다.
- [x] Train / Validation / Test 역할을 구분했습니다.
- [x] 모델을 Validation에서 선택했습니다.
- [x] Threshold를 Validation에서 선택했습니다.
- [x] Final Test 전에 선택을 고정했습니다.
- [x] Precision / Recall / F1을 해석했습니다.
- [x] FP/FN의 의미를 설명했습니다.
- [x] Public prediction에서 식별자를 제거했습니다.
- [x] Validation Evidence를 확인했습니다.
- [x] random split의 한계를 기록했습니다.
- [x] 예측 패턴을 원인처럼 단정하지 않았습니다.
- [x] 최종 Notebook URL을 제출합니다.